# Critical Path for Data Quality, Consistency Alignment, and Pre-Clustering Feature Selection

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("../EDAgroup5/data/sample_grouped_dataset-1.csv")

## **Systemic Null Value Resolution**

Define and execute an explicit, domain-justified rule for every missing value pattern identified in the panel dataset:

## Structural NULL handling

General rule applied to **every** imputed column: add a boolean
`_was_missing` flag **before** filling. 

In [3]:
imp_log = {}          
df_c = df.copy()

def impute(col, value, rule, add_flag=True, fill_label=None):
    n = int(df_c[col].isna().sum())
    if add_flag:
        df_c[col + "_was_missing"] = df_c[col].isna()
    df_c[col] = df_c[col].fillna(value)
    if fill_label is None:
        fill_label = value if np.isscalar(value) else "<vectorised>"
    imp_log[col] = {"rule": rule, "fill": fill_label,
                    "n_filled": n, "flag": f"{col}_was_missing" if add_flag else None}

### Group E — historical context

* **`*_avg_prev3m`** — user has no 3-month history yet. Neutral fill = the
  **current-month analogue** when one exists (best available estimate of the
  user's recent level), else the column median. A single `is_history_seed`
  flag marks these rows.
* **`*_delta_m`** — change vs. 3m average is genuinely *0* information when there
  is no baseline → fill **0**.
* **`spend_velocity_index_m`** — ratio-type index → fill **1.0** (neutral "no
  acceleration"), flagged.

In [4]:
LAG_ANALOGUE = {                       # prev3m feature  ->  current-month analogue
    "tx_count_avg_prev3m":           "tx_count_m",
    "tx_amount_avg_prev3m":          "tx_amount_avg_m",
    "oxxo_ratio_avg_prev3m":         "oxxo_tx_ratio_m",
    "cash_funding_ratio_avg_prev3m": "cash_funding_ratio_m",
    "ecommerce_ratio_avg_prev3m":    "ecommerce_tx_ratio_m",
}
df_c["is_history_seed"] = df_c["tx_count_avg_prev3m"].isna()   # single explanatory flag

for lag_col, cur_col in LAG_ANALOGUE.items():
    fill = df_c[cur_col].astype(float)
    fill = fill.fillna(fill.median())
    impute(lag_col, fill, f"first-3-months: fill with current-month analogue '{cur_col}', "
                          f"else median", add_flag=False,
           fill_label=f"current-month analogue '{cur_col}' (residual -> median)")

for d in ["tx_count_delta_m","tx_amount_avg_delta_m","oxxo_ratio_delta_m",
          "cash_funding_ratio_delta_m","ecommerce_ratio_delta_m"]:
    impute(d, 0.0, "no 3m baseline -> delta carries no signal -> 0", add_flag=False)

impute("spend_velocity_index_m", 1.0, "neutral 'no acceleration' index value")
print("Group E done. history-seed rows:", int(df_c['is_history_seed'].sum()))

Group E done. history-seed rows: 83025


## Group F — friction & recovery

* **`recovery_avg_min_m`** NULL ⇒ *there was nothing to recover from* (no decline,
  or no post-decline success). This is **not** a large latency — fill **0** and
  expose `had_decline_m` + `recovery_avg_min_m_was_missing` so the model can tell
  "recovered instantly" from "never needed to".
* **`top_rejection_reason_m`** NULL ⇒ no decline → explicit category **`"NONE"`**.
* **`oxxo_decline_ratio_m` / `same_merchant_retry_ratio_m`** — already 0-filled in
  source for no-decline users; leave as is.

In [7]:
df_c["had_decline_m"] = df_c["decline_count_m"].fillna(0).gt(0)
impute("recovery_avg_min_m", 0.0,
       "NULL = no decline or no post-decline success -> 0 min, keep flag + had_decline_m")

df_c["top_rejection_reason_m"] = (
    df_c["top_rejection_reason_m"].astype('category').cat.add_categories(["NONE"]).fillna("NONE"))
imp_log["top_rejection_reason_m"] = {"rule": "NULL = no declined tx -> category 'NONE'",
                                     "fill": "NONE", "n_filled": int(df["top_rejection_reason_m"].isna().sum()),
                                     "flag": None}
print(df_c["top_rejection_reason_m"].value_counts().head(6))

top_rejection_reason_m
ISUFFOPENTOBUY        222156
NONE                  209244
CARDBLOCKEDDECLINE     20081
CARDBLOCKEDFRAUD       13691
NOACTIONTAKEN          11510
INVALIDCVV2/CVC2        9159
Name: count, dtype: int64
